In [36]:
from google.colab import drive

In [37]:
RUN_FROM_COLAB = True

In [38]:
import sklearn
print(sklearn.__version__)

1.6.1


In [39]:
import xgboost
print(xgboost.__version__)

3.4.1


In [40]:
import joblib
print(joblib.__version__)

1.5.3


In [41]:
if RUN_FROM_COLAB:
  drive.mount('/content/drive')
  DATA_PATH = '/content/drive/MyDrive/spam-classification-fastapi/data'
  MODELS_PATH = '/content/drive/MyDrive/spam-classification-fastapi/models'
else:
  DATA_PATH = '/spam-classification-fastapi/data/spam.csv'
  MODELS_PATH = '/spam-classification-fastapi/models'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [42]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
df = pd.read_csv(DATA_PATH+'/spam.csv', usecols=['v1', 'v2'], encoding='ISO-8859-1').rename(columns={'v1':'label', 'v2':'text'})
df

,label,text
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives around here though"
...,...,...
5567,spam,"This is the 2nd time we have tried 2 contact u. U have won the å£750 Pound prize. 2 claim is easy, call 087187272008 NOW1! Only 10p per minute. BT-national-rate."
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other suggestions?"
5570,ham,The guy did some bitching but I acted like i'd be interested in buying something else next week and he gave it to us for free


In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   5572 non-null   object
 1   text    5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [44]:
from sklearn.preprocessing import LabelEncoder

In [45]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])
df

,label,text
0,0,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives around here though"
...,...,...
5567,1,"This is the 2nd time we have tried 2 contact u. U have won the å£750 Pound prize. 2 claim is easy, call 087187272008 NOW1! Only 10p per minute. BT-national-rate."
5568,0,Will Ì_ b going to esplanade fr home?
5569,0,"Pity, * was in mood for that. So...any other suggestions?"
5570,0,The guy did some bitching but I acted like i'd be interested in buying something else next week and he gave it to us for free


In [46]:
df2 = pd.read_csv(DATA_PATH + '/spam_ham_dataset.csv', usecols=['text', 'label_num']).rename(columns={"label_num":"label"})
df2

,text,label
0,"Subject: enron methanol ; meter # : 988291\r\nthis is a follow up to the note i gave you on monday , 4 / 3 / 00 { preliminary\r\nflow data provided by daren } .\r\nplease override pop ' s daily volume { presently zero } to reflect daily\r\nactivity you can obtain from gas control .\r\nthis change is needed asap for economics purposes .",0
1,"Subject: hpl nom for january 9 , 2001\r\n( see attached file : hplnol 09 . xls )\r\n- hplnol 09 . xls",0
2,"Subject: neon retreat\r\nho ho ho , we ' re around to that most wonderful time of the year - - - neon leaders retreat time !\r\ni know that this time of year is extremely hectic , and that it ' s tough to think about anything past the holidays , but life does go on past the week of december 25 through january 1 , and that ' s what i ' d like you to think about for a minute .\r\non the calender that i handed out at the beginning of the fall semester , the retreat was scheduled for the weekend of january 5 - 6 . but because of a youth ministers conference that brad and dustin are connected with that week , we ' re going to change the date to the following weekend , january 12 - 13 . now comes the part you need to think about .\r\ni think we all agree that it ' s important for us to get together and have some time to recharge our batteries before we get to far into the spring semester , but it can be a lot of trouble and difficult for us to get away without kids , etc . so , brad came up with a potential alternative for how we can get together on that weekend , and then you can let me know which you prefer .\r\nthe first option would be to have a retreat similar to what we ' ve done the past several years . this year we could go to the heartland country inn ( www . . com ) outside of brenham . it ' s a nice place , where we ' d have a 13 - bedroom and a 5 - bedroom house side by side . it ' s in the country , real relaxing , but also close to brenham and only about one hour and 15 minutes from here . we can golf , shop in the antique and craft stores in brenham , eat dinner together at the ranch , and spend time with each other . we ' d meet on saturday , and then return on sunday morning , just like what we ' ve done in the past .\r\nthe second option would be to stay here in houston , have dinner together at a nice restaurant , and then have dessert and a time for visiting and recharging at one of our homes on that saturday evening . this might be easier , but the trade off would be that we wouldn ' t have as much time together . i ' ll let you decide .\r\nemail me back with what would be your preference , and of course if you ' re available on that weekend . the democratic process will prevail - - majority vote will rule ! let me hear from you as soon as possible , preferably by the end of the weekend . and if the vote doesn ' t go your way , no complaining allowed ( like i tend to do ! )\r\nhave a great weekend , great golf , great fishing , great shopping , or whatever makes you happy !\r\nbobby",0
3,"Subject: photoshop , windows , office . cheap . main trending\r\nabasements darer prudently fortuitous undergone\r\nlighthearted charm orinoco taster\r\nrailroad affluent pornographic cuvier\r\nirvin parkhouse blameworthy chlorophyll\r\nrobed diagrammatic fogarty clears bayda\r\ninconveniencing managing represented smartness hashish\r\nacademies shareholders unload badness\r\ndanielson pure caffein\r\nspaniard chargeable levin\r\n",1
4,"Subject: re : indian springs\r\nthis deal is to book the teco pvr revenue . it is my understanding that teco\r\njust sends us a check , i haven ' t received an answer as to whether there is a\r\npredermined price associated with this deal or if teco just lets us know what\r\nwe are giving . i can continue to chase this deal down if you need .",0
...,...,...
5166,"Subject: put the 10 on the ft\r\nthe transport volumes decreased from 25000 to 10000 . all 10000 should be on\r\ncontract 012 - 41991 - 203 .\r\nthanks ,\r\nami\r\n- - - - - - -

In [47]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5171 entries, 0 to 5170
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    5171 non-null   object
 1   label   5171 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 80.9+ KB


In [48]:
df2['text'] = df2['text'].apply(lambda s: s[s.find('\n') + 1:])

In [49]:
df2

,text,label
0,"this is a follow up to the note i gave you on monday , 4 / 3 / 00 { preliminary\r\nflow data provided by daren } .\r\nplease override pop ' s daily volume { presently zero } to reflect daily\r\nactivity you can obtain from gas control .\r\nthis change is needed asap for economics purposes .",0
1,( see attached file : hplnol 09 . xls )\r\n- hplnol 09 . xls,0
2,"ho ho ho , we ' re around to that most wonderful time of the year - - - neon leaders retreat time !\r\ni know that this time of year is extremely hectic , and that it ' s tough to think about anything past the holidays , but life does go on past the week of december 25 through january 1 , and that ' s what i ' d like you to think about for a minute .\r\non the calender that i handed out at the beginning of the fall semester , the retreat was scheduled for the weekend of january 5 - 6 . but because of a youth ministers conference that brad and dustin are connected with that week , we ' re going to change the date to the following weekend , january 12 - 13 . now comes the part you need to think about .\r\ni think we all agree that it ' s important for us to get together and have some time to recharge our batteries before we get to far into the spring semester , but it can be a lot of trouble and difficult for us to get away without kids , etc . so , brad came up with a potential alternative for how we can get together on that weekend , and then you can let me know which you prefer .\r\nthe first option would be to have a retreat similar to what we ' ve done the past several years . this year we could go to the heartland country inn ( www . . com ) outside of brenham . it ' s a nice place , where we ' d have a 13 - bedroom and a 5 - bedroom house side by side . it ' s in the country , real relaxing , but also close to brenham and only about one hour and 15 minutes from here . we can golf , shop in the antique and craft stores in brenham , eat dinner together at the ranch , and spend time with each other . we ' d meet on saturday , and then return on sunday morning , just like what we ' ve done in the past .\r\nthe second option would be to stay here in houston , have dinner together at a nice restaurant , and then have dessert and a time for visiting and recharging at one of our homes on that saturday evening . this might be easier , but the trade off would be that we wouldn ' t have as much time together . i ' ll let you decide .\r\nemail me back with what would be your preference , and of course if you ' re available on that weekend . the democratic process will prevail - - majority vote will rule ! let me hear from you as soon as possible , preferably by the end of the weekend . and if the vote doesn ' t go your way , no complaining allowed ( like i tend to do ! )\r\nhave a great weekend , great golf , great fishing , great shopping , or whatever makes you happy !\r\nbobby",0
3,abasements darer prudently fortuitous undergone\r\nlighthearted charm orinoco taster\r\nrailroad affluent pornographic cuvier\r\nirvin parkhouse blameworthy chlorophyll\r\nrobed diagrammatic fogarty clears bayda\r\ninconveniencing managing represented smartness hashish\r\nacademies shareholders unload badness\r\ndanielson pure caffein\r\nspaniard chargeable levin\r\n,1
4,"this deal is to book the teco pvr revenue . it is my understanding that teco\r\njust sends us a check , i haven ' t received an answer as to whether there is a\r\npredermined price associated with this deal or if teco just lets us know what\r\nwe are giving . i can continue to chase this deal down if you need .",0
...,...,...
5166,"the transport volumes decreased from 25000 to 10000 . all 10000 should be on\r\ncontract 012 - 41991 - 203 .\r\nthanks ,\r\nami\r\n- - - - - - - - - - - - - - - - - - - - - - forwarded by ami chokshi / corp / enron on 08 / 31 / 2000\r\n10 : 54 am - - - - - - - - - - - - - - - - - - - - - - - - - - -\r\nroyal _ b _ edmondson @ reliantenergy . com on 08 / 31 / 2000 10 : 47 : 37 am\r\nto : 

In [50]:
df = pd.concat([df, df2], ignore_index=True)

In [51]:
df

,label,text
0,0,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives around here though"
...,...,...
10738,0,"the transport volumes decreased from 25000 to 10000 . all 10000 should be on\r\ncontract 012 - 41991 - 203 .\r\nthanks ,\r\nami\r\n- - - - - - - - - - - - - - - - - - - - - - forwarded by ami chokshi / corp / enron on 08 / 31 / 2000\r\n10 : 54 am - - - - - - - - - - - - - - - - - - - - - - - - - - -\r\nroyal _ b _ edmondson @ reliantenergy . com on 08 / 31 / 2000 10 : 47 : 37 am\r\nto : ami _ chokshi @ enron . com\r\ncc :\r\nsubject : put the 10 on the ft\r\n( see attached file : hpl - sept . xls )\r\n- hpl - sept . xls"
10739,0,"hpl can ' t take the extra 15 mmcf / d over the weekend . we ' ll try next week ,\r\nbut for now the nom will stay at 60 mmcf / d , with redeliveries as they have\r\nbeen\r\n-\r\n50 into pg & e , 7 from fcv , and 3 at carthage .\r\n- - - - - - - - - - - - - - - - - - - - - - forwarded by bruce mcmills / ftworth / pefs / pec on\r\n03 / 03 / 2000\r\n09 : 42 am - - - - - - - - - - - - - - - - - - - - - - - - - - -\r\nbruce mcmills\r\n03 / 03 / 2000 09 : 10 am\r\nto : dfarmer @ enron . com , briley @ enron . com , stacey . neuweiler @ enron . com\r\ncc : chad w . cass / gcs / cec / pec @ pec , william e . speckels / gcs / cec / pec @ pec , donna\r\nc . spencer / gcs / cec / pec @ pec , michael r . cherry / easttexas / pefs / pec @ pec ,\r\ndarrel f . bane / easttexas / pefs / pec @ pec\r\nsubject : 3 / 4 / 2000 and following noms\r\nthis is to nominate 75 , 000 mmbtu / d into eastrans for 3 / 4 / 2000 and following .\r\nwe will deliver 50 , 000 into pg & e , 7 , 000 from fuel cotton valley ( continue\r\n750\r\nmmbtu / d sale ) ,\r\nand 18 , 000 mmbtu / d into your cartwheel agreement at carthage ."
10740,0,">\r\n>\r\njulie , as i mention earlier we hope to start the unit this afternoon but\r\nare still experiencing difficulties . i will keep you informed . thanks .\r\nricky a . archer\r\nfuel supply\r\n700 louisiana , suite 2700\r\nhouston , texas 77002\r\n713 - 830 - 8659 direct\r\n713 - 830 - 8722 fax\r\n- calpine daily gas nomination 1 . doc\r\n- calpine daily gas nomination 1 . doc"
10741,0,"attached are the worksheets for august 2000 activity . there are three\r\ndifferent worksheets { 2 - supply & 1 - market } .\r\nthe market worksheet is preliminary and will continuously be updated\r\nthroughout the month .\r\nthe supply worksheets capture all "" buybacks and the relevant pricing data .\r\nthese three worksheets can be found in two separate files . o :\r\nlogistics / robert lloyd / buydeaug 2000 . xls . . . . . . . . . . . . . supply\r\no : logistics / ken\r\nsorry for the delay in providing you ' ll this data ."


In [52]:
df.groupby('label').count()

,text
label,
0,8497
1,2246


In [53]:
import spacy
!python -m spacy download en_core_web_md
nlp = spacy.load('en_core_web_md')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 44.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [54]:
import re
import os
def normalize_text(text):
    text = text.lower()
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\W+', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    doc = nlp(text)
    text = ' '.join([token.lemma_ for token in doc if not token.is_stop])

    text = re.sub(r'\b\w\b', ' ', text) # single characters

    return text

if not os.path.isfile(DATA_PATH+'/normalized_text.pkl'):
    df['text'] = df['text'].apply(normalize_text)
    pd.to_pickle(df['text'], DATA_PATH+'/normalized_text.pkl')
else:
    df['text'] = pd.read_pickle(DATA_PATH+'/normalized_text.pkl')
df

,label,text
0,0,jurong point crazy available bugis great world la buffet cine get amore wat
1,0,ok lar joke wif oni
2,1,free entry wkly comp win fa cup final tkts st text fa receive entry question std txt rate apply
3,0,dun early hor
4,0,nah don think go usf live
...,...,...
10738,0,transport volume decrease contract thank ami forward ami chokshi corp enron royal edmondson reliantenergy com ami chokshi enron com cc subject ft attach file hpl sept xls hpl sept xls
10739,0,hpl extra mmcf weekend ll try week nom stay mmcf redeliverie pg fcv carthage forward bruce mcmills ftworth pef pec bruce mcmill dfarmer enron com briley enron com stacey neuweiler enron com cc chad cass gcs cec pec pec william speckels gcs cec pec pec donna spencer gcs cec pec pec michael cherry easttexas pef pec pec darrel bane easttexas pef pec pec subject follow nom nominate mmbtu eastran follow deliver pg fuel cotton valley continue mmbtu sale mmbtu cartwheel agreement carthage
10740,0,julie mention early hope start unit afternoon experience difficulty inform thank ricky archer fuel supply louisiana suite houston texas direct fax calpine daily gas nomination doc calpine daily gas nomination doc
10741,0,attach worksheet august activity different worksheet supply market market worksheet preliminary continuously update month supply worksheet capture buyback relevant pricing datum worksheet find separate file logistics robert lloyd buydeaug xls supply logistic ken sorry delay provide ll data


In [55]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from scipy import sparse
import joblib

In [56]:
if not os.path.isfile(DATA_PATH+'/BoW.npz'):
    cv = CountVectorizer(ngram_range=(1, 2))
    bag_of_words = cv.fit_transform(df['text'])
    joblib.dump(cv, MODELS_PATH+'/count_vectorizer.joblib')
    sparse.save_npz(DATA_PATH+'/BoW.npz', bag_of_words)
else:
    cv = joblib.load(MODELS_PATH+'/count_vectorizer.joblib')
    bag_of_words = sparse.load_npz(DATA_PATH+'/BoW.npz')
bag_of_words

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 720664 stored elements and shape (10743, 268963)>

In [57]:
if not os.path.isfile(DATA_PATH+'/tfidf.npz'):
    tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    tfidf = tfidf_vectorizer.fit_transform(df['text'])
    joblib.dump(tfidf_vectorizer, MODELS_PATH+'/tfidf_vectorizer.joblib')
    sparse.save_npz(DATA_PATH+'/tfidf.npz', tfidf)
else:
    tfidf_vectorizer = joblib.load(MODELS_PATH+'/tfidf_vectorizer.joblib')
    tfidf = sparse.load_npz(DATA_PATH+'/tfidf.npz')
tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 720664 stored elements and shape (10743, 268963)>

In [58]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import classification_report
from sklearn.pipeline import make_pipeline

In [59]:
X_train, X_test, y_train, y_test = train_test_split(bag_of_words, df['label'], test_size=0.25, random_state=42)

In [60]:
if not os.path.isfile(MODELS_PATH+'/nb_pipeline.joblib'):
    gcv = GridSearchCV(estimator=MultinomialNB(), param_grid={'alpha': [1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 2.0], 'fit_prior': [True, False]}, cv=5, scoring='f1')
    gcv.fit(X_train, y_train)
    nb = gcv.best_estimator_
    nb_pipeline = make_pipeline(cv, nb)
    joblib.dump(nb_pipeline, MODELS_PATH+'/nb_pipeline.joblib')
else:
    nb_pipeline = joblib.load(MODELS_PATH+'/nb_pipeline.joblib')

y_pred = nb_pipeline[-1].predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.97      0.97      2099
           1       0.89      0.91      0.90       587

    accuracy                           0.96      2686
   macro avg       0.93      0.94      0.94      2686
weighted avg       0.96      0.96      0.96      2686



In [61]:
from sklearn.linear_model import LogisticRegression

In [62]:
X_train, X_test, y_train, y_test = train_test_split(tfidf, df['label'], test_size=0.25, random_state=42)

In [63]:
if not os.path.isfile(MODELS_PATH+'/logreg_pipeline.joblib'):
    gcv = GridSearchCV(estimator=LogisticRegression(), param_grid={'penalty':['l2'], 'C':[1.0, 0.1, 0.5, 10] ,'solver':['lbfgs', ], 'max_iter':[100, 500, 1000]}, cv=5, scoring='f1')
    gcv.fit(X_train, y_train)
    logreg = gcv.best_estimator_
    logreg_pipeline = make_pipeline(tfidf_vectorizer, logreg)
    joblib.dump(logreg_pipeline, MODELS_PATH+'/logreg_pipeline.joblib')
else:
    logreg_pipeline = joblib.load(MODELS_PATH+'/logreg_pipeline.joblib')

y_pred = logreg_pipeline[-1].predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      1.00      0.96      2099
           1       0.98      0.74      0.84       587

    accuracy                           0.94      2686
   macro avg       0.96      0.87      0.90      2686
weighted avg       0.94      0.94      0.94      2686



In [64]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

In [65]:
param_distributions = {
    "max_depth": [3, 5, 7, 9],
    "min_child_weight": [1, 3, 5, 10],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "n_estimators": [200, 500, 1000],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "gamma": [0, 0.1, 0.5, 1, 5],
    "reg_alpha": [0, 0.01, 0.1, 1],
    "reg_lambda": [1, 2, 5, 10],
}
if not os.path.isfile(MODELS_PATH+'/xgboost_pipeline.joblib'):
    rscv = RandomizedSearchCV(estimator=XGBClassifier(device='cuda', tree_method="hist", random_state=42),
                              param_distributions=param_distributions, random_state=42, cv=5, scoring='f1', n_iter=20, n_jobs=1)
    rscv.fit(X_train, y_train)
    xgbst = rscv.best_estimator_
    xgboost_pipeline = make_pipeline(tfidf_vectorizer, xgbst)
    joblib.dump(xgboost_pipeline, MODELS_PATH+'/xgboost_pipeline.joblib')
else:
    xgboost_pipeline = joblib.load(MODELS_PATH+'/xgboost_pipeline.joblib')

y_pred = xgboost_pipeline[-1].predict(X_test)
print(classification_report(y_test, y_pred))

/usr/lib/python3.13/pickle.py:1754: UserWarning: [10:19:25] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:439: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [10:19:25] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [10:19:25] WARNING: /__w/xgboost/xgboost/src/context.cc:218: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  setstate(state)


              precision    recall  f1-score   support

           0       0.95      0.98      0.97      2099
           1       0.93      0.83      0.88       587

    accuracy                           0.95      2686
   macro avg       0.94      0.91      0.92      2686
weighted avg       0.95      0.95      0.95      2686



In [66]:
from sklearn.ensemble import VotingClassifier

In [67]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.25, random_state=42)

In [68]:
X_train

,text
7299,discount med right home valium xanax weight reduction day delivery license physician office visit private confidential discreet http www legend drug biz usa interested thank http www legend drug biz unsubscribe ddd sundew dimension clint amend churchyard concordant sprig biblical bronco postwar medicine coin smithereen commissary denumerable acrylate torrent aniseikonic alabaster bimodal fructify protophyta gloat archibald faunaphdpowers dragoon giovanni suntanne bakelite sacrificial wharf sociometry stolid butyl termite goa creosote moll splendid alcestis send onlooke wellesley multipliable britannicadodresponsive
4071,loan purpose bad credit tenant welcome noworriesloan com
6758,agree typo memo rate hour show eileen ponton david avila lsp enserch tu charlie stone texas utility tu melissa jones texas utilities tu hpl scheduling enron com liz bellamy enron com cc subject nom actual flow hrs hrs nom mcf mmbtu
724,world run maybe feel admit mad correction let life run world run let run
5009,way rencontre meet mountain not
...,...
5734,gbhzivjwl
5191,sorry ll later
5390,not joke seriously told
860,work go min


In [69]:
if not os.path.isfile(MODELS_PATH+'/voting_classifier.joblib'):
    vc = VotingClassifier(estimators=[('naive_bayes', nb_pipeline), ('logistic_regression', logreg_pipeline), ('xgboost', xgboost_pipeline)], voting='soft')
    gscv = GridSearchCV(estimator=vc, param_grid={'weights': [[1, 1, 2], [1, 1, 3], [1, 2, 1], [1, 2, 2]]}, cv=5, scoring='f1')
    gscv.fit(X_train, y_train)
    voting_classifier = gscv.best_estimator_
    joblib.dump(voting_classifier, MODELS_PATH+'/voting_classifier.joblib')
else:
  voting_classifier = joblib.load(MODELS_PATH+'/voting_classifier.joblib')

y_pred = voting_classifier.predict(X_test)
print(classification_report(y_test, y_pred))

/usr/lib/python3.13/pickle.py:1754: UserWarning: [10:19:36] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:439: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [10:19:36] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [10:19:36] WARNING: /__w/xgboost/xgboost/src/context.cc:218: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  setstate(state)


              precision    recall  f1-score   support

           0       0.96      0.99      0.98      2099
           1       0.98      0.84      0.90       587

    accuracy                           0.96      2686
   macro avg       0.97      0.92      0.94      2686
weighted avg       0.96      0.96      0.96      2686



In [70]:
voting_classifier.weights

[1, 2, 1]